In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [4]:
data=pd.read_csv("Churn_Modelling.csv")

In [14]:
data=data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

load_encoder_gender = LabelEncoder()
data['Gender'] = load_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geography = OneHotEncoder(handle_unknown='ignore')
geography_encoded = onehot_encoder_geography.fit_transform(data[['Geography']]).toarray()
geography_df = pd.DataFrame(geography_encoded, columns=onehot_encoder_geography.get_feature_names_out(['Geography']))
data = pd.concat([data.drop('Geography', axis=1), geography_df], axis=1)

x=data.drop('Exited', axis=1)
y=data['Exited']

KeyError: "['RowNumber', 'CustomerId', 'Surname'] not found in axis"

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [8]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('encoder_gender.pkl', 'wb') as f:
    pickle.dump(load_encoder_gender, f)

with open('onehot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geography, f)

In [47]:
def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(x_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [51]:
##create a KerasClassifier wrapper for the model

model=KerasClassifier(layers=1, neurons=32,build_fn=create_model, verbose=0)


In [52]:
#define the grid search parameters

param_grid = {
    'neurons': [16, 32, 64],
    'layers': [1, 2],
    
    'epochs': [50, 100]
}

In [53]:
#perform grid search

grid=GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result=grid.fit(x_train, y_train)

#print the best param

print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

/opt/anaconda3/envs/evn/lib/python3.10/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/evn/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/opt/anaconda3/envs/evn/lib/python3.10/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/opt/anaconda3/envs/evn/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to 

Best: 0.856624 using {'epochs': 50, 'layers': 1, 'neurons': 16}
